# Multi-Modal Evaluation with fasteval

This notebook demonstrates how to evaluate **multi-modal AI systems** using fasteval:

1. **Text evaluation** — correctness and relevance of LLM responses
2. **Vision-language** — evaluating image understanding with GPT-4o
3. **Speech / WER** — deterministic word-error-rate (no API key needed)
4. **Image generation** — evaluating generated images for quality and prompt adherence
5. **Multi-modal RAG** — evaluating answers grounded in text + image context

Every cell in this notebook is **runnable**. Cells that call LLM-as-judge metrics require an OpenAI API key.

---

## Setup

In [ ]:
# Install fasteval with all multi-modal features + openai client
!pip install -q "fasteval[multimodal]" openai

In [ ]:
import os

# --- Set your OpenAI API key (required for LLM-as-judge metrics) ---
#
# Option 1: Set it directly (uncomment and paste your key)
# os.environ["OPENAI_API_KEY"] = "sk-..."
#
# Option 2: Colab Secrets (recommended)
#   1. Click the key icon in the left sidebar
#   2. Add a secret named OPENAI_API_KEY with your key
#   3. Toggle "Notebook access" on
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("API key loaded from Colab Secrets.")
except (ImportError, Exception):
    if os.environ.get("OPENAI_API_KEY"):
        print("API key found in environment.")
    else:
        print(
            "WARNING: OPENAI_API_KEY not set.\n"
            "Text and vision examples will fail. "
            "WER examples (Section 3) work without a key."
        )

In [ ]:
import fasteval as fe
from openai import OpenAI

# OpenAI client for calling LLMs in our examples
openai_client = OpenAI()

print(f"fasteval ready — default provider will use OPENAI_API_KEY for LLM-as-judge.")

---

## 1. Text Evaluation — Correctness & Relevance

The simplest case: call an LLM, then let fasteval judge whether the response is correct and relevant.

- `@fe.correctness` — Does the response match the expected answer semantically?
- `@fe.relevance` — Is the response relevant to the input question?

In [ ]:
def ask_llm(question: str) -> str:
    """Call GPT-4o-mini with a question and return the response."""
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": question}],
        temperature=0.0,
    )
    return response.choices[0].message.content


@fe.correctness(threshold=0.7)
@fe.relevance(threshold=0.7)
def test_geography_qa():
    question = "What is the capital of France and what is it known for?"
    response = ask_llm(question)
    print(f"Question : {question}")
    print(f"Response : {response[:200]}..." if len(response) > 200 else f"Response : {response}")

    return fe.score(
        actual_output=response,
        expected_output="The capital of France is Paris. It is known for the Eiffel Tower, the Louvre, and French cuisine.",
        input=question,
    )


result = test_geography_qa()
print(f"\nPassed: {result.passed}  |  Score: {result.aggregate_score:.2f}")
for m in result.metric_results:
    print(f"  {m.metric_name}: {m.score:.2f} (threshold {m.threshold})")

---

## 2. Vision-Language Models

Evaluate how well a vision-language model understands an image.

We use GPT-4o (which supports vision) to describe a public image, then `@fe.image_understanding` judges the response against an expected answer.

> **Note:** This requires a vision-capable model. GPT-4o works well.

In [ ]:
# A public image of a mountain landscape (Unsplash, free to use)
IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/e/e7/Everest_North_Face_toward_Base_Camp_Tibet_Luca_Galuzzi_2006.jpg/1280px-Everest_North_Face_toward_Base_Camp_Tibet_Luca_Galuzzi_2006.jpg"


def ask_vision(image_url: str, prompt: str) -> str:
    """Call GPT-4o with an image and a text prompt."""
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": image_url}},
                ],
            }
        ],
        temperature=0.0,
        max_tokens=300,
    )
    return response.choices[0].message.content


@fe.image_understanding(threshold=0.7)
def test_mountain_image():
    prompt = "What mountain is shown in this image? Describe the scene briefly."
    response = ask_vision(IMAGE_URL, prompt)
    print(f"Prompt   : {prompt}")
    print(f"Response : {response[:250]}..." if len(response) > 250 else f"Response : {response}")

    return fe.score(
        actual_output=response,
        expected_output=(
            "The image shows Mount Everest, the tallest mountain in the world. "
            "The scene shows the north face of Everest with a rocky, barren "
            "foreground leading to the snow-capped peak."
        ),
        image=IMAGE_URL,
        input=prompt,
    )


result = test_mountain_image()
print(f"\nPassed: {result.passed}  |  Score: {result.aggregate_score:.2f}")
for m in result.metric_results:
    print(f"  {m.metric_name}: {m.score:.2f} (threshold {m.threshold})")

---

## 3. Speech Recognition — Word Error Rate

The `@fe.word_error_rate` metric is **deterministic** — it computes WER by comparing the predicted transcript against the ground truth. No API key or LLM call is needed.

The score is `1 - WER`, so a threshold of `0.95` means WER must be below 5%.

In [ ]:
# Simulated ground truth and transcription output
ground_truth = (
    "The quarterly earnings report showed a fifteen percent increase in revenue "
    "driven primarily by strong performance in the cloud services division"
)

# A good transcription (minor errors)
good_transcript = (
    "The quarterly earnings report showed a fifteen percent increase in revenue "
    "driven primarily by strong performance in the cloud services division"
)

# A poor transcription (several errors)
poor_transcript = (
    "The quarterly earning report show a fifty percent increase in revenue "
    "driven primarily by stong preformance in the clod services division"
)


@fe.word_error_rate(threshold=0.95)
def test_good_transcript():
    return fe.score(
        actual_output=good_transcript,
        expected_output=ground_truth,
    )


@fe.word_error_rate(threshold=0.95)
def test_poor_transcript():
    return fe.score(
        actual_output=poor_transcript,
        expected_output=ground_truth,
    )


# --- Good transcript (should pass) ---
print("=== Good Transcript ===")
result_good = test_good_transcript()
print(f"Passed: {result_good.passed}  |  Score: {result_good.aggregate_score:.2f}")
for m in result_good.metric_results:
    print(f"  {m.metric_name}: {m.score:.2f} (threshold {m.threshold})")

# --- Poor transcript (should fail) ---
print("\n=== Poor Transcript ===")
try:
    result_poor = test_poor_transcript()
    print(f"Passed: {result_poor.passed}  |  Score: {result_poor.aggregate_score:.2f}")
except fe.EvaluationFailedError as e:
    print(f"EvaluationFailedError: {e}")
    print("(This is expected — the poor transcript has too many errors.)")

---

## 4. Image Generation Evaluation

Evaluate the quality and prompt adherence of a generated image.

Here we use a **publicly available image** as a stand-in for a generated image, then evaluate it with:
- `@fe.image_quality` — Is the image technically good? (composition, clarity, etc.)
- `@fe.prompt_adherence` — Does the image match the generation prompt?

In [ ]:
from fasteval import GeneratedImage, ImageInput

# Using a public Wikimedia image of a red sports car as our "generated" image
GENERATED_IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/42/2017_Alfa_Romeo_Giulia_Quadrifoglio_%28Type_952%29_sedan_%282018-10-29%29_01.jpg/1280px-2017_Alfa_Romeo_Giulia_Quadrifoglio_%28Type_952%29_sedan_%282018-10-29%29_01.jpg"

GENERATION_PROMPT = "A red sports car parked on a street"

# Wrap it as a GeneratedImage
generated_img = GeneratedImage(
    image=ImageInput(source=GENERATED_IMAGE_URL),
    prompt=GENERATION_PROMPT,
    model="dall-e-3",  # Simulated model name
)


@fe.image_quality(threshold=0.6)
@fe.prompt_adherence(threshold=0.7)
def test_image_generation():
    print(f"Prompt: {GENERATION_PROMPT}")
    print(f"Image : {GENERATED_IMAGE_URL[:80]}...")

    return fe.score(
        generated_image=generated_img,
        input=GENERATION_PROMPT,
    )


result = test_image_generation()
print(f"\nPassed: {result.passed}  |  Score: {result.aggregate_score:.2f}")
for m in result.metric_results:
    print(f"  {m.metric_name}: {m.score:.2f} (threshold {m.threshold})")

---

## 5. Multi-Modal RAG

Evaluate a RAG system that answers questions using both text and image context.

`@fe.multimodal_faithfulness` checks whether the answer is grounded in the provided multi-modal context (text chunks + images).

In [ ]:
from fasteval import ImageInput

# Simulated retrieved context from a financial report
retrieved_text_chunks = [
    "Q3 2024 Financial Summary: Total revenue was $4.2M, a 15% increase year-over-year.",
    "Cloud services contributed $2.8M (67% of total revenue), up from $2.1M in Q3 2023.",
    "Operating expenses were $3.1M, resulting in operating income of $1.1M.",
]

# A chart image showing the revenue breakdown (using a public chart image)
CHART_IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/b/b5/Tesco_PLC_revenue_by_geography%2C_2011-2018.png/640px-Tesco_PLC_revenue_by_geography%2C_2011-2018.png"

# Simulated RAG answer
rag_answer = "Q3 revenue was $4.2M, representing a 15% year-over-year increase. Cloud services was the primary driver at $2.8M."


@fe.multimodal_faithfulness(threshold=0.7)
def test_financial_qa():
    question = "What was Q3 revenue and what drove the growth?"
    print(f"Question : {question}")
    print(f"Answer   : {rag_answer}")
    print(f"Context  : {len(retrieved_text_chunks)} text chunks + 1 chart image")

    return fe.score(
        actual_output=rag_answer,
        expected_output="Q3 revenue was $4.2M, a 15% increase driven by cloud services.",
        input=question,
        context=retrieved_text_chunks,
        images=[ImageInput(source=CHART_IMAGE_URL)],
    )


result = test_financial_qa()
print(f"\nPassed: {result.passed}  |  Score: {result.aggregate_score:.2f}")
for m in result.metric_results:
    print(f"  {m.metric_name}: {m.score:.2f} (threshold {m.threshold})")

---

## Data Models Reference

fasteval provides typed Pydantic models for multi-modal inputs. Here is a quick reference.

In [ ]:
from fasteval import ImageInput, AudioInput, GeneratedImage

# --- ImageInput ---
# From a URL
img_url = ImageInput(source="https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/320px-Camponotus_flavomarginatus_ant.jpg")
print(f"ImageInput(url)  — is_url: {img_url.is_url()}")

# From a file path (does not need to exist for model construction)
img_file = ImageInput(source="chart.png", alt_text="Q3 Revenue Chart", metadata={"page": 5})
print(f"ImageInput(file) — is_file: {img_file.is_file()}, alt_text: {img_file.alt_text}")

# --- AudioInput ---
audio = AudioInput(
    source="speech.wav",
    duration_seconds=120.5,
    transcript="Optional ground truth transcript",
)
print(f"AudioInput       — duration: {audio.duration_seconds}s, has transcript: {audio.transcript is not None}")

# --- GeneratedImage ---
gen_img = GeneratedImage(
    image=img_url,
    prompt="A close-up photo of an ant",
    model="dall-e-3",
    seed=42,
)
print(f"GeneratedImage   — model: {gen_img.model}, prompt: {gen_img.prompt[:40]}...")

---

## `fe.score()` — Full Parameter Reference

The extended `fe.score()` signature supports all multi-modal inputs:

```python
fe.score(
    actual_output=response,
    expected_output="...",
    input=query,
    context=["chunk1", "chunk2"],

    # Multi-modal parameters
    image="chart.png",                    # Single image (path, URL, or ImageInput)
    images=["img1.png", "img2.png"],      # Multiple images
    audio="recording.mp3",                # Audio file (path, URL, or AudioInput)
    generated_image=gen_img,              # GeneratedImage for image-gen eval
    expected_fields={"total": "$1,234"},  # For OCR / extraction metrics
)
```

---

## Available Metrics

| Category | Metrics | Type |
|----------|---------|------|
| **Vision-Language** | `image_understanding`, `ocr_accuracy`, `chart_interpretation`, `visual_grounding`, `image_faithfulness` | LLM-as-judge |
| **Audio/Speech** | `word_error_rate`, `character_error_rate`, `match_error_rate` | Deterministic |
| **Audio/Speech** | `transcription_accuracy`, `speaker_diarization`, `audio_sentiment` | LLM-as-judge |
| **Image Generation** | `image_quality`, `prompt_adherence`, `safety_check`, `aesthetic_score`, `clip_score` | LLM-as-judge |
| **Multi-Modal RAG** | `multimodal_faithfulness`, `table_extraction`, `figure_reference`, `cross_modal_coherence`, `document_understanding` | LLM-as-judge |

For full documentation, visit the [fasteval docs](https://fasteval.dev/docs).